SHRIHARIHARAN S

24MCS1058

In [12]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import evaluate
import torch

# Step 1: Load the FLAN-T5 model and tokenizer
model_name = "google/flan-t5-small"  # we can choose a larger model like flan-t5-base or flan-t5-large
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step 2: Prepare a dataset (using CNN/DailyMail as an example)
dataset = load_dataset("cnn_dailymail", "3.0.0", split={"train": "train[:1%]", "test": "test[:1%]"})

def preprocess_data(batch):
    # Tokenize inputs and labels
    inputs = tokenizer(batch["article"], max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(batch["highlights"], max_length=64, truncation=True, padding="max_length")
    inputs["labels"] = [
        label if mask else -100 for label, mask in zip(labels["input_ids"], labels["attention_mask"])
    ]
    return inputs

# Apply preprocessing
dataset = dataset.map(preprocess_data, batched=True, remove_columns=["article", "highlights", "id"])

# Step 3: Prepare for fine-tuning
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# Define training arguments
training_args = TrainingArguments(
    output_dir="./flan-t5-summarization",  # Directory for saving checkpoints
    evaluation_strategy="steps",         # Evaluate at each step
    eval_steps=500,                       # Number of steps for evaluation
    save_steps=500,                       # Save the model checkpoint at each step
    per_device_train_batch_size=2,        # Reduced batch size to avoid OOM
    per_device_eval_batch_size=2,         # Reduced batch size for evaluation
    gradient_accumulation_steps=4,        # Simulate a larger batch size
    num_train_epochs=3,
    learning_rate=1e-5,                   # Reduced learning rate
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,                     # Log more frequently
    save_total_limit=2,
    max_grad_norm=1.0,                    # Gradient clipping to prevent overflow
    fp16=False                            # Disable mixed precision temporarily
)

# Define a data collator
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Step 4: Fine-tune the model using Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=None,  # Deprecated, tokenizer removed
    data_collator=data_collator
)

# Start training
try:
    torch.cuda.empty_cache()  # Clear CUDA memory
    trainer.train()
except torch.cuda.OutOfMemoryError:
    print("Out of Memory! Try reducing batch size or sequence length further.")

# Step 5: Evaluate the model
# Use the evaluate library for metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

# Function to compute metrics
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE
    rouge_scores = rouge.compute(predictions=preds, references=labels)
    bleu_score = bleu.compute(predictions=preds, references=[[l] for l in labels])

    return {
        "rouge": rouge_scores,
        "bleu": bleu_score["bleu"]
    }

# Evaluate on the evaluation dataset
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

Map:   0%|          | 0/2871 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-12-82b5ef25c22c>:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss
500,13.835000,3.694179
1000,12.570700,3.042836


Evaluation Results: {'eval_loss': 3.035639524459839, 'eval_runtime': 1.6634, 'eval_samples_per_second': 69.135, 'eval_steps_per_second': 34.868, 'epoch': 3.0}
